In [33]:
!nvidia-smi

Tue Apr  7 11:37:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P0             31W /   70W |    4479MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [34]:
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

In [35]:
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install transformers accelerate

Found existing installation: transformers 5.5.0
Uninstalling transformers-5.5.0:
  Successfully uninstalled transformers-5.5.0
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
Using cached transformers-5.5.0-py3-none-any.whl (10.2 MB)
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)


### Purpose of accelerate
1. Ease of multi device training
2. Mixed precision
3. Zero Redundancy Optimizer
4. Offload to CPU/SSD


In [1]:
from transformers import pipeline, set_seed
from datasets import load_dataset, load_from_disk
import matplotlib.pyplot as plt
from datasets import load_dataset
import pandas as pd

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

import nltk
from nltk.tokenize import sent_tokenize

nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [2]:
import torch
device="cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [6]:
from transformers import AutoTokenizer, PegasusForConditionalGeneration, AutoModelForSeq2SeqLM
model=PegasusForConditionalGeneration.from_pretrained("google/pegasus-xsum")
tokenizer=AutoTokenizer.from_pretrained("google/pegasus-xsum")

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-xsum
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
ARTICLE_TO_SUMMARIZE=(
    "Artificial Intelligence has revolutionized how we process and condense vast amounts of textual information. Text summarization, once limited to simple keyword extraction, now leverages sophisticated transformer models that understand context, semantics, and even stylistic nuances."
)

In [8]:
inputs=tokenizer(ARTICLE_TO_SUMMARIZE,max_length=1024,return_tensors="pt")
summary_ids=model.generate(inputs["input_ids"])
tokenizer.batch_decode(summary_ids,skip_special_tokens=True,clean_up_tokenization_spaces=False)[0]

'Text summarization has the potential to transform the way we communicate.'

In [9]:
from transformers import AutoModelForSeq2SeqLM,AutoTokenizer
device="cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

# Fine Tuning task

In [10]:
model="google/pegasus-cnn_dailymail"
tokenizer=AutoTokenizer.from_pretrained(model)
model_pegasus=AutoModelForSeq2SeqLM.from_pretrained(model).to(device)

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
# !wget https://github.com/krishnaik06/datasets/raw/refs/heads/main/summarizer-data.zip
# !unzip summarizer-data.zip

--2026-04-07 11:44:38--  https://github.com/krishnaik06/datasets/raw/refs/heads/main/summarizer-data.zip
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/krishnaik06/datasets/refs/heads/main/summarizer-data.zip [following]
--2026-04-07 11:44:39--  https://raw.githubusercontent.com/krishnaik06/datasets/refs/heads/main/summarizer-data.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7903594 (7.5M) [application/zip]
Saving to: ‘summarizer-data.zip.2’

summarizer-data.zip 100%[===================>]   7.54M  --.-KB/s    in 0.07s   

2026-04-07 11:44:39 (115 MB/s) - ‘summarizer-data.zip.2’ saved [79

In [13]:
data_set=load_from_disk('samsum_dataset')

In [14]:
data_set

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14732
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
})

In [15]:
split_lengths=[len(data_set[split]) for split in data_set]
print(f"Splits length: {split_lengths}")
print(f"Features: {data_set['train'].column_names}")

Splits length: [14732, 819, 818]
Features: ['id', 'dialogue', 'summary']


In [16]:
print("\n Dialogue")
print(data_set["test"][1]["dialogue"])
print("\n Summary")
print(data_set["test"][1]["summary"])


 Dialogue
Eric: MACHINE!
Rob: That's so gr8!
Eric: I know! And shows how Americans see Russian ;)
Rob: And it's really funny!
Eric: I know! I especially like the train part!
Rob: Hahaha! No one talks to the machine like that!
Eric: Is this his only stand-up?
Rob: Idk. I'll check.
Eric: Sure.
Rob: Turns out no! There are some of his stand-ups on youtube.
Eric: Gr8! I'll watch them now!
Rob: Me too!
Eric: MACHINE!
Rob: MACHINE!
Eric: TTYL?
Rob: Sure :)

 Summary
Eric and Rob are going to watch a stand-up on youtube.


In [22]:
def convert_examples_to_features(example_batch):
  input_encodings=tokenizer(example_batch['dialogue'],max_length=1024,truncation=True)
  target_encodings=tokenizer(example_batch['summary'],max_length=1024,truncation=True)
  return{
      "input_ids":input_encodings["input_ids"],
      "attention_mask": input_encodings['attention_mask'],
      'labels': target_encodings['input_ids']
  }

In [23]:
dataset_samsum_pt=data_set.map(convert_examples_to_features,batched=True)

Map:   0%|          | 0/14732 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

In [24]:
dataset_samsum_pt

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 14732
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 819
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 818
    })
})

In [27]:
from transformers import DataCollatorForSeq2Seq
seq2seq_data_collator=DataCollatorForSeq2Seq(tokenizer,model=model_pegasus)
# DataCollatorForSeq2Seq is a special data collator designed for sequence-to-sequence models(e.g. Pegasus,T5,BART) that helps in preparing batches of data for training

In [31]:
from transformers import TrainingArguments, Trainer
trainer_args=TrainingArguments(
    output_dir="pegasus-samsum",num_train_epochs=1,warmup_steps=500,
    per_device_train_batch_size=1,per_device_eval_batch_size=1,
    weight_decay=0.01,logging_steps=10,
    eval_strategy="steps",eval_steps=5,save_steps=1e6,
    gradient_accumulation_steps=16
)

In [33]:
trainer=Trainer(model=model_pegasus,args=trainer_args,
                data_collator=seq2seq_data_collator,train_dataset=dataset_samsum_pt["test"],
                eval_dataset=dataset_samsum_pt["validation"])

In [34]:
trainer.train()

Step,Training Loss,Validation Loss
5,No log,2.693733
10,50.272324,2.685140
15,50.272324,2.670286
20,52.218121,2.649561
25,52.218121,2.622468
30,49.079025,2.589887
35,49.079025,2.551646
40,47.101608,2.512502
45,47.101608,2.473528
50,47.226511,2.430511


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=52, training_loss=48.390093289888824, metrics={'train_runtime': 814.0981, 'train_samples_per_second': 1.006, 'train_steps_per_second': 0.064, 'total_flos': 314203859361792.0, 'train_loss': 48.390093289888824, 'epoch': 1.0})

In [35]:
def generate_batch_sized_chunks(list_of_elements,batch_size):
  '''
  Splits dataset into smaller batches that we can process simultaneously
  Yields successive batch-sized chunks from list_of_elements
  '''
  for i in range(0,len(list_of_elements),batch_size):
    yield list_of_elements[i:i+batch_size]

In [37]:
!pip install tqdm

In [48]:
import tqdm
def calculate_metric_on_test_ds(dataset,metric,model,tokenizer,batch_size=16,device=device,column_text="article",column_summary="highlights"):
  article_batches=list(generate_batch_sized_chunks(dataset[column_text],batch_size))
  target_batches=list(generate_batch_sized_chunks(dataset[column_summary],batch_size))
  for article_batch,target_batch in tqdm.tqdm(zip(article_batches,target_batches),total=len(article_batches)):
    input=tokenizer(article_batch,max_length=1024,truncation=True,padding="max_length",return_tensors="pt")
    summaries=model.generate(input_ids=input["input_ids"].to(device),
                             attention_mask=input["attention_mask"].to(device),
                             length_penalty=0.8,num_beams=8,max_length=128)
    decoded_summaries=[tokenizer.decode(s,skip_special_tokens=True,clean_up_tokenization_spaces=True) for s in summaries]
    decoded_summaries=[d.replace(""," ") for d in decoded_summaries]
    metric.add_batch(predictions=decoded_summaries,references=target_batch)
  score=metric.compute()
  return score

In [41]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00


In [44]:
import evaluate
rough_metric=evaluate.load('rouge')
rough_name=["rouge1","rouge2","rougeL","rougeLsum"]


In [49]:
score=calculate_metric_on_test_ds(
    data_set['test'][:10],rough_metric,trainer.model,tokenizer,batch_size=2,column_text="dialogue",column_summary="summary"
)

100%|██████████| 5/5 [00:28<00:00,  5.65s/it]


In [50]:
rough_dir={rn:score[rn] for rn in rough_name}
import pandas as pd
pd.DataFrame(rough_dir,index=[f"pegasus"])


,rouge1,rouge2,rougeL,rougeLsum
pegasus,0.020023,0.0,0.019792,0.019789


In [51]:
## As we have trained it for only one epoch we have this as an result for more epochs we would be able to generate better result more the rouge is close to 1
# More better accuracy the model provides
model_pegasus.save_pretrained("pegasus-samsum-model")
tokenizer.save_pretrained("tokenizer")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('tokenizer/tokenizer_config.json', 'tokenizer/tokenizer.json')